# Dataset 2 — Embedding threshold sensitivity (graphsage_v1_64 + MLP tuned)

Uses the **per-p** `graphsage_v1_64` embeddings (re-extracted for each threshold; see
`03_embeddings/dataset_2/03_embeddings_p_thresholds.ipynb`). For each p the selected MLP is
re-fit on that p's embeddings + target with the same 70/15/15 stratified split.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / 'src').exists() and (p / 'requirements.txt').exists():
            return p
    raise FileNotFoundError('Project root not found.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from src.models.ml_train_and_store import load_model, random_split, CLASSICAL_FEATURE_CANDIDATES

DATASET = 'dataset_2'
APPROACH = 'embedding'
TARGET_COL = 'log_systemic_risk_label'
P_LIST = ['p0', 'p5', 'p10', 'p15', 'p20', 'p25', 'p30', 'p35', 'p40']

TARGETS_DIR = PROJECT_ROOT / 'src' / 'datasets' / 'dataset_2' / 'targets'
SRC = PROJECT_ROOT / 'src' / 'data' / 'embeddings' / 'dataset_2' / 'feature_based' / 'graphsage_v1_64_dataset2_dataset.parquet'
MODEL_PATH  = PROJECT_ROOT / 'src' / 'models' / 'dataset_2/05_a/graphsage_v1_64/MLP_(tuned).joblib'
OUT_DIR     = PROJECT_ROOT / 'src' / 'data' / 'predictions' / 'dataset_2/embedding_threshold'
OUT_DIR.mkdir(parents=True, exist_ok=True)

model = load_model(MODEL_PATH)
print('model:', MODEL_PATH.name, '| approach:', APPROACH)

model: MLP_(tuned).joblib | approach: embedding


## Apply the selected model across thresholds

In [2]:
def load_p(p):
    # Per-p embeddings (graphsage_v1_64 re-extracted for each p; target already merged in).
    # p0 uses the baseline file; p5..p40 use the per-threshold files from
    # 03_embeddings/dataset_2/03_embeddings_p_thresholds.ipynb.
    name = SRC if p == 'p0' else SRC.with_name(f'graphsage_v1_64_dataset2_{p}_dataset.parquet')
    df = pd.read_parquet(name)
    ecols = [c for c in df.columns if c.startswith('emb_')]
    return df[['bank_id'] + ecols + [TARGET_COL]].dropna(subset=[TARGET_COL]).reset_index(drop=True), ecols

In [3]:
rows, preds = [], []
for p in P_LIST:
    df, fcols = load_p(p)
    tr, va, te = random_split(df, TARGET_COL)            # 70/15/15 stratified
    m = clone(model).fit(tr[fcols], tr[TARGET_COL])      # same selected model, refit at this threshold
    row = {'p': p}
    for split_name, sdf in [('train', tr), ('validation', va), ('test', te)]:
        yp = m.predict(sdf[fcols])
        row[f'{split_name}_rmse'] = mean_squared_error(sdf[TARGET_COL], yp) ** 0.5
        row[f'{split_name}_mae']  = mean_absolute_error(sdf[TARGET_COL], yp)
        pf = sdf[['bank_id', TARGET_COL]].copy()
        pf['prediction'] = yp; pf['split'] = split_name; pf['p'] = p
        pf['dataset'] = DATASET; pf['approach'] = APPROACH
        preds.append(pf)
    row['val/train_rmse'] = round(row['validation_rmse'] / row['train_rmse'], 2)
    row['val/train_mae']  = round(row['validation_mae'] / row['train_mae'], 2)
    rows.append(row)

metrics = pd.DataFrame(rows)
predictions = pd.concat(preds, ignore_index=True)
metrics.to_csv(OUT_DIR / 'metrics.csv', index=False)
predictions.to_csv(OUT_DIR / 'predictions.csv', index=False)
print('saved ->', OUT_DIR)
display(metrics.round(3))

saved -> /Users/rubenmarques/Documents/Repositórios/Thesis/src/data/predictions/dataset_2/embedding_threshold


,p,train_rmse,train_mae,validation_rmse,validation_mae,test_rmse,test_mae,val/train_rmse,val/train_mae
0,p0,0.137,0.054,0.142,0.063,0.167,0.060,1.04,1.15
1,p5,0.144,0.061,0.182,0.076,0.171,0.067,1.26,1.25
2,p10,0.159,0.064,0.147,0.075,0.207,0.075,0.92,1.17
3,p15,0.170,0.071,0.180,0.086,0.174,0.073,1.06,1.22
4,p20,0.179,0.070,0.177,0.081,0.180,0.070,0.99,1.16
5,p25,0.166,0.067,0.193,0.080,0.166,0.080,1.16,1.18
6,p30,0.170,0.079,0.256,0.097,0.193,0.095,1.50,1.22
7,p35,0.165,0.070,0.313,0.102,0.220,0.082,1.89,1.47
8,p40,0.170,0.080,0.251,0.103,0.258,0.094,1.48,1.30
